[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevin-blasiak-curtin/ISYS2001-Archive/blob/main/Module%2009%20-%20OOP_Gradio/building-an-interface-step-by-step.ipynb)

# Building an Interface, Step by Step

Each cell below is a complete, working app. You are not writing it from scratch. The code is already here so you can experiment with it.

The routine for every step is the same:

1. Run the cell and see what appears.
2. Read the short note above it, which points at one or two things to change.
3. Change them, run the cell again, and watch how the interface responds.

The skill worth building is not typing the code. It is noticing *why* a change has the effect it does. Every step is pre-loaded with a few sample transactions, so you will see something straight away without having to add anything first.

When you are done, there is a short prompt at the bottom linking this back to your own Assessment 2 project.

## Setup

Run these two cells once. The first installs Gradio. The second defines the two classes we will use all the way through, `Transaction` and `FinanceTracker`, and a small helper that hands back a tracker already holding a few sample transactions.

You do not need to change anything in the setup cells.

In [ ]:
# Install Gradio (run once). Gradio moves quickly, so you could pin a version
# here for a stable semester, e.g. !pip install -q gradio==6.*
!pip install -q gradio

In [ ]:
import gradio as gr
from datetime import datetime

import matplotlib
matplotlib.use("Agg")  # a non-interactive backend, so charts hand cleanly to Gradio
import matplotlib.pyplot as plt


class Transaction:
    """A single financial transaction."""

    def __init__(self, description, amount, category="Uncategorised"):
        self.description = description
        self.amount = amount
        self.category = category
        self.timestamp = datetime.now()

    def __repr__(self):
        return f"Transaction('{self.description}', ${self.amount:.2f}, {self.category})"


class FinanceTracker:
    """Holds a collection of transactions and answers questions about them."""

    def __init__(self):
        self.transactions = []

    def add(self, transaction):
        self.transactions.append(transaction)

    def total(self):
        return sum(t.amount for t in self.transactions)

    def by_category(self):
        totals = {}
        for t in self.transactions:
            totals[t.category] = totals.get(t.category, 0) + t.amount
        return totals


def make_tracker():
    """Return a fresh tracker with a few sample transactions already in it."""
    t = FinanceTracker()
    t.add(Transaction("Coffee", -4.50, "Food"))
    t.add(Transaction("Bus fare", -3.20, "Transport"))
    t.add(Transaction("Salary", 3000.00, "Income"))
    t.add(Transaction("Groceries", -67.80, "Food"))
    return t

## Step 1: The starting point

The simplest useful app: a box for a description, a box for an amount, a button, and a line that confirms what happened. Clicking the button runs `add_txn`, which builds a `Transaction` and hands back a message.

**Try changing:**

- The button text `"Add Transaction"`.
- The `value=0.0` default on the amount box.
- The heading in `gr.Markdown`.

Run the cell again after each change.

In [ ]:
tracker = make_tracker()

def add_txn(description, amount):
    tracker.add(Transaction(description, amount))
    return f"Added: {description} for ${amount:.2f}"

with gr.Blocks() as demo:
    gr.Markdown("## Transaction Logger")

    description = gr.Textbox(label="Description", placeholder="e.g. Coffee")
    amount = gr.Number(label="Amount", value=0.0)
    add_button = gr.Button("Add Transaction")
    confirmation = gr.Textbox(label="Confirmation", interactive=False)

    add_button.click(fn=add_txn, inputs=[description, amount], outputs=confirmation)

demo.launch()

## Step 2: Add a category

Every transaction should be tagged. We add one new object, a `gr.Dropdown`, and pass its value into `add_txn` as a third input. Notice the dropdown is built from a plain Python list.

**Try changing:**

- Add a category to the `CATEGORIES` list, such as `"Health"`, and see it appear in the dropdown.
- Change `value="Food"` to a different starting category.
- Change the dropdown `label`.

In [ ]:
tracker = make_tracker()

CATEGORIES = ["Food", "Transport", "Income", "Bills", "Entertainment"]

def add_txn(description, amount, category):
    tracker.add(Transaction(description, amount, category))
    return f"Added: {description} (${amount:.2f}) in {category}"

with gr.Blocks() as demo:
    gr.Markdown("## Transaction Logger")

    description = gr.Textbox(label="Description", placeholder="e.g. Coffee")
    amount = gr.Number(label="Amount", value=0.0)
    category = gr.Dropdown(label="Category", choices=CATEGORIES, value="Food")
    add_button = gr.Button("Add Transaction")
    confirmation = gr.Textbox(label="Confirmation", interactive=False)

    add_button.click(fn=add_txn, inputs=[description, amount, category], outputs=confirmation)

demo.launch()

## Step 3: Show the transactions in a table

So far we can add transactions but cannot see them. We add a `gr.Dataframe` to display the whole collection. The important idea: `add_txn` now returns *two* things, a message and the updated table, and the button sends each to a different output object.

**Try changing:**

- The column names in `headers`.
- The order of the columns in `table_data` (swap description and category).
- What happens if you remove `table` from the `outputs` list? Add a transaction and see.

In [ ]:
tracker = make_tracker()

CATEGORIES = ["Food", "Transport", "Income", "Bills", "Entertainment"]

def table_data():
    return [[t.description, round(t.amount, 2), t.category] for t in tracker.transactions]

def add_txn(description, amount, category):
    tracker.add(Transaction(description, amount, category))
    return f"Added: {description}", table_data()

with gr.Blocks() as demo:
    gr.Markdown("## Transaction Logger")

    description = gr.Textbox(label="Description", placeholder="e.g. Coffee")
    amount = gr.Number(label="Amount", value=0.0)
    category = gr.Dropdown(label="Category", choices=CATEGORIES, value="Food")
    add_button = gr.Button("Add Transaction")
    confirmation = gr.Textbox(label="Confirmation", interactive=False)
    table = gr.Dataframe(headers=["Description", "Amount", "Category"], value=table_data())

    add_button.click(fn=add_txn, inputs=[description, amount, category],
                     outputs=[confirmation, table])

demo.launch()

## Step 4: Add a running summary

A table is data. A summary is insight. We add a `summary_text` function that asks the `FinanceTracker` for its total and its category breakdown, and show it in a multi-line box that updates on every add.

**Try changing:**

- The wording `"Total balance"`.
- Add a line that also reports how many transactions there are (hint: `len(tracker.transactions)`).
- Add an expense in a new category and watch the breakdown change.

In [ ]:
tracker = make_tracker()

CATEGORIES = ["Food", "Transport", "Income", "Bills", "Entertainment"]

def table_data():
    return [[t.description, round(t.amount, 2), t.category] for t in tracker.transactions]

def summary_text():
    lines = [f"Total balance: ${tracker.total():.2f}", ""]
    for cat, amount in tracker.by_category().items():
        lines.append(f"{cat}: ${amount:.2f}")
    return "\n".join(lines)

def add_txn(description, amount, category):
    tracker.add(Transaction(description, amount, category))
    return f"Added: {description}", table_data(), summary_text()

with gr.Blocks() as demo:
    gr.Markdown("## Transaction Logger")

    description = gr.Textbox(label="Description", placeholder="e.g. Coffee")
    amount = gr.Number(label="Amount", value=0.0)
    category = gr.Dropdown(label="Category", choices=CATEGORIES, value="Food")
    add_button = gr.Button("Add Transaction")
    confirmation = gr.Textbox(label="Confirmation", interactive=False)
    table = gr.Dataframe(headers=["Description", "Amount", "Category"], value=table_data())
    summary = gr.Textbox(label="Summary", value=summary_text(), lines=7, interactive=False)

    add_button.click(fn=add_txn, inputs=[description, amount, category],
                     outputs=[confirmation, table, summary])

demo.launch()

## Step 5: Add a chart

Numbers are easier to read as a picture. We add a `gr.Plot` and a `spending_chart` function that draws expenses by category with matplotlib, the same library you used for the stock charts in Module 08. The chart updates on every add, just like the table and summary.

**Try changing:**

- The bar `color` (it is set to the unit's gold, `"#B8933B"`).
- Swap `ax.bar` for `ax.barh` to get a horizontal chart.
- The chart title.

In [ ]:
tracker = make_tracker()

CATEGORIES = ["Food", "Transport", "Income", "Bills", "Entertainment"]

def table_data():
    return [[t.description, round(t.amount, 2), t.category] for t in tracker.transactions]

def summary_text():
    lines = [f"Total balance: ${tracker.total():.2f}", ""]
    for cat, amount in tracker.by_category().items():
        lines.append(f"{cat}: ${amount:.2f}")
    return "\n".join(lines)

def spending_chart():
    spending = {}
    for t in tracker.transactions:
        if t.amount < 0:
            spending[t.category] = spending.get(t.category, 0) + abs(t.amount)
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(list(spending.keys()), list(spending.values()), color="#B8933B")
    ax.set_title("Spending by category")
    ax.set_ylabel("Amount ($)")
    fig.tight_layout()
    return fig

def add_txn(description, amount, category):
    tracker.add(Transaction(description, amount, category))
    return f"Added: {description}", table_data(), summary_text(), spending_chart()

with gr.Blocks() as demo:
    gr.Markdown("## Transaction Logger")

    description = gr.Textbox(label="Description", placeholder="e.g. Coffee")
    amount = gr.Number(label="Amount", value=0.0)
    category = gr.Dropdown(label="Category", choices=CATEGORIES, value="Food")
    add_button = gr.Button("Add Transaction")
    confirmation = gr.Textbox(label="Confirmation", interactive=False)
    table = gr.Dataframe(headers=["Description", "Amount", "Category"], value=table_data())
    summary = gr.Textbox(label="Summary", value=summary_text(), lines=7, interactive=False)
    chart = gr.Plot(value=spending_chart())

    add_button.click(fn=add_txn, inputs=[description, amount, category],
                     outputs=[confirmation, table, summary, chart])

demo.launch()

## Step 6: Organise it into tabs

The screen is getting busy. We do not add any new features here. We reorganise the same components into three `gr.Tab` sections, so entering data, viewing it, and analysing it each have their own space. The button still lives on the first tab but updates outputs on the others.

**Try changing:**

- Rename the tabs.
- Move the `summary` box onto the same tab as the chart.
- Reorder the tabs and see how the app opens differently.

In [ ]:
tracker = make_tracker()

CATEGORIES = ["Food", "Transport", "Income", "Bills", "Entertainment"]

def table_data():
    return [[t.description, round(t.amount, 2), t.category] for t in tracker.transactions]

def summary_text():
    lines = [f"Total balance: ${tracker.total():.2f}", ""]
    for cat, amount in tracker.by_category().items():
        lines.append(f"{cat}: ${amount:.2f}")
    return "\n".join(lines)

def spending_chart():
    spending = {}
    for t in tracker.transactions:
        if t.amount < 0:
            spending[t.category] = spending.get(t.category, 0) + abs(t.amount)
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(list(spending.keys()), list(spending.values()), color="#B8933B")
    ax.set_title("Spending by category")
    ax.set_ylabel("Amount ($)")
    fig.tight_layout()
    return fig

def add_txn(description, amount, category):
    tracker.add(Transaction(description, amount, category))
    return f"Added: {description}", table_data(), summary_text(), spending_chart()

with gr.Blocks() as demo:
    gr.Markdown("## Finance Tracker")

    with gr.Tab("Add"):
        description = gr.Textbox(label="Description", placeholder="e.g. Coffee")
        amount = gr.Number(label="Amount", value=0.0)
        category = gr.Dropdown(label="Category", choices=CATEGORIES, value="Food")
        add_button = gr.Button("Add Transaction")
        confirmation = gr.Textbox(label="Confirmation", interactive=False)

    with gr.Tab("Transactions"):
        table = gr.Dataframe(headers=["Description", "Amount", "Category"], value=table_data())

    with gr.Tab("Analysis"):
        summary = gr.Textbox(label="Summary", value=summary_text(), lines=7, interactive=False)
        chart = gr.Plot(value=spending_chart())

    add_button.click(fn=add_txn, inputs=[description, amount, category],
                     outputs=[confirmation, table, summary, chart])

demo.launch()

## One last change, and then over to you

**Share it.** Change the final line to `demo.launch(share=True)` and run it again. Gradio gives you a public link you can send to someone so they can open your app in their own browser. Handy for showing a friend or a tutor.

**The thread running through all of this:** every step added one more object. A dropdown is an object, a table is an object, a chart is an object. Your `Transaction` and `FinanceTracker` are objects too. Building an app is composing objects that hand values to one another.

**For your Assessment 2 project**, jot a few lines in your Developer's Diary:

- What are the equivalents of `Transaction` and `FinanceTracker` in your own solution?
- Which of these components would your app actually need, and which would it not?

You do not have to build the whole thing today. Knowing which objects you need, and which parts of this you would reuse, is the useful first step.